# ShiftLog-Gym Eval + Publish

Compare base vs trained adapters on held-out variants, generate plots/artifacts, and push outputs to Hugging Face.

In [ ]:
# Colab setup: clone repo + install it so `shiftlog_gym` is importable.
import os

REPO_URL = "https://github.com/Chirag0096/ShiftLog-Gym.git"
REPO_DIR = "ShiftLog-Gym"

if os.path.exists(REPO_DIR):
    %cd {REPO_DIR}
else:
    !git clone {REPO_URL}
    %cd {REPO_DIR}

!pip -q install -e .


In [ ]:
!pip -q install transformers peft accelerate huggingface_hub pandas matplotlib seaborn


In [ ]:
import ast
import json
import re
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from huggingface_hub import HfApi, create_repo, notebook_login, upload_folder
from peft import AutoPeftModelForCausalLM
from transformers import AutoModelForCausalLM, AutoTokenizer

from shiftlog_gym.scenarios import FAMILIES
from shiftlog_gym.simulator import ShiftLogSimulator
from shiftlog_gym.training import AVAILABLE_TOOLS, TEST_VARIANTS, rollout_prompted_policy, summarize_episode, write_artifacts

sns.set_theme(style='whitegrid')


In [ ]:
BASE_MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
ADAPTER_PATH = 'outputs/grpo-stage-c'
OUTPUT_DIR = Path('artifacts/eval_publish')
HF_MODEL_REPO = 'Chirag0096/shiftlog-gym-qwen2.5-3b-memory-policy'
MAX_NEW_TOKENS = 180


In [ ]:
def load_generation_stack(model_name_or_path, adapter=False):
    if adapter:
        model = AutoPeftModelForCausalLM.from_pretrained(model_name_or_path, torch_dtype='auto', device_map='auto')
        tokenizer = AutoTokenizer.from_pretrained(model.base_model.model.name_or_path)
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name_or_path, torch_dtype='auto', device_map='auto')
        tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
    tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
    return model, tokenizer


def extract_action_dict(text, fallback_service='payments-api'):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return {'tool': 'inspect_service', 'arguments': {'service': fallback_service}}
    candidate = match.group(0)
    try:
        parsed = json.loads(candidate)
    except json.JSONDecodeError:
        parsed = ast.literal_eval(candidate)
    if parsed.get('tool') not in AVAILABLE_TOOLS:
        return {'tool': 'inspect_service', 'arguments': {'service': fallback_service}}
    return parsed


def build_policy(model, tokenizer):
    def _policy(observation, transcript):
        prompt = f"""
You are the primary SRE on call in ShiftLog-Gym.
Choose exactly one JSON tool call using this schema: {{"tool": str, "arguments": dict}}.
Prefer read_shift_log before acting on repeated or linked incidents.
Available tools: {', '.join(AVAILABLE_TOOLS)}.
Observation:\n{observation}\n
Recent transcript:\n{json.dumps(transcript[-2:], indent=2)}
"""
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        decoded = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return extract_action_dict(decoded)
    return _policy


In [ ]:
base_model, base_tokenizer = load_generation_stack(BASE_MODEL_NAME, adapter=False)
base_policy = build_policy(base_model, base_tokenizer)

trained_model = None
trained_tokenizer = None
trained_policy = None
if Path(ADAPTER_PATH).exists():
        trained_model, trained_tokenizer = load_generation_stack(ADAPTER_PATH, adapter=True)
        trained_policy = build_policy(trained_model, trained_tokenizer)
        print('Loaded trained adapter from', ADAPTER_PATH)
else:
        print('Adapter path not found; base-only evaluation will run.')


In [ ]:
def evaluate_policy(policy_name, policy_fn, variant_indices=TEST_VARIANTS, seed=0):
    rows = []
    memory_events = []
    tool_events = []
    trajectory_rows = []
    for family in FAMILIES:
        for variant_index in variant_indices:
            simulator = ShiftLogSimulator()
            simulator.reset(seed=seed, family=family, variant_index=variant_index)
            transcript = rollout_prompted_policy(simulator, policy_fn, max_steps=18)
            artifacts = summarize_episode(simulator, f'{policy_name}-{family}-{variant_index}', 'test', seed, variant_index)
            rows.append(artifacts.episode_row)
            memory_events.extend(artifacts.memory_events)
            tool_events.extend(artifacts.tool_timeline)
            for step_index, item in enumerate(transcript):
                trajectory_rows.append({
                    'policy_name': policy_name,
                    'family': family,
                    'variant_index': variant_index,
                    'step_index': step_index,
                    'tool': item['action'].get('tool'),
                    'tool_response': item['tool_response'],
                })
    return rows, memory_events, tool_events, trajectory_rows


base_rows, base_memory, base_tools, base_trajectories = evaluate_policy('base', base_policy)
all_rows = list(base_rows)
all_memory = list(base_memory)
all_tools = list(base_tools)
all_trajectories = list(base_trajectories)

if trained_policy is not None:
    trained_rows, trained_memory, trained_tools, trained_trajectories = evaluate_policy('trained', trained_policy)
    all_rows.extend(trained_rows)
    all_memory.extend(trained_memory)
    all_tools.extend(trained_tools)
    all_trajectories.extend(trained_trajectories)

eval_df = pd.DataFrame(all_rows)
eval_df


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_artifacts(OUTPUT_DIR, all_rows, all_memory, all_tools)
pd.DataFrame(all_trajectories).to_csv(OUTPUT_DIR / 'trajectory_steps.csv', index=False)
print('Wrote eval artifacts to', OUTPUT_DIR)


In [ ]:
metrics_to_plot = [
    'weighted_reward',
    'R_recall',
    'recall_before_action_rate',
    'linked_incident_success_rate',
    'contradiction_rate',
]

plots_dir = OUTPUT_DIR / 'plots'
plots_dir.mkdir(exist_ok=True)

for metric in metrics_to_plot:
    fig, axis = plt.subplots(figsize=(8, 4))
    sns.barplot(data=eval_df, x='family', y=metric, hue='episode_name', ax=axis)
    axis.set_title(metric)
    axis.tick_params(axis='x', rotation=25)
    fig.tight_layout()
    fig.savefig(plots_dir / f'{metric}.png', dpi=160)
    plt.show()


In [ ]:
trajectory_df = pd.DataFrame(all_trajectories)
trajectory_df.head(20)


## Publish to Hugging Face

This uploads the LoRA adapter and eval artifacts, and writes a cautious model-card summary aligned with the environment's current short-horizon scope.

In [ ]:
PUBLISH_TO_HF = False

MODEL_CARD_TEXT = """
# ShiftLog-Gym Memory Policy Adapter

This adapter is trained for professional-domain memory-policy learning in ShiftLog-Gym.

Claim language:
- domain-specific professional RL environment for memory management
- causal recall behavior improves after RL
- does not claim persistent memory is solved

Current scope:
- short-horizon current environment (3 incidents per scenario)
- optimized for free-T4 Colab experimentation
""".strip()

if PUBLISH_TO_HF:
    notebook_login()
    create_repo(HF_MODEL_REPO, exist_ok=True, repo_type='model')
    (OUTPUT_DIR / 'README.md').write_text(MODEL_CARD_TEXT, encoding='utf-8')
    upload_folder(repo_id=HF_MODEL_REPO, folder_path=str(OUTPUT_DIR), repo_type='model')
    if Path(ADAPTER_PATH).exists():
        upload_folder(repo_id=HF_MODEL_REPO, folder_path=ADAPTER_PATH, repo_type='model', path_in_repo='adapter')
    print('Published to', HF_MODEL_REPO)
else:
    print('Publishing skipped. Set PUBLISH_TO_HF = True when you are ready to upload.')
